In [0]:
from pyspark.sql import functions as sf
from delta.tables import DeltaTable

silver_path = "/Volumes/Weather/default/02_silver"

df_bronze = spark.read.format("Delta").load("/Volumes/weather/default/01_bronze")

df_silver = df_bronze.withColumn("weather_desc", sf.col("weather")[0]["description"])

df_silver = df_silver.withColumn("weather_main", sf.col("weather")[0]["main"])

df_silver = df_silver.withColumn(
    "observation_timestamp_utc", sf.to_timestamp(sf.from_unixtime(sf.col("dt")))
)

df_silver = df_silver.withColumn(
    "observation_timestamp_local",
    sf.to_timestamp(sf.from_unixtime(sf.col("dt") + sf.col("timezone"))),
)

df_silver = df_silver.withColumn(
    "sunrise_utc", sf.to_timestamp(sf.from_unixtime(sf.col("`sys.sunrise`")))
)

df_silver = df_silver.withColumn(
    "sunset_utc", sf.to_timestamp(sf.from_unixtime(sf.col("`sys.sunset`")))
)

df_silver = df_silver.withColumn(
    "sunrise_local",
    sf.to_timestamp(sf.from_unixtime(sf.col("`sys.sunrise`") + sf.col("timezone"))),
)

df_silver = df_silver.withColumn(
    "sunset_local",
    sf.to_timestamp(sf.from_unixtime(sf.col("`sys.sunset`") + sf.col("timezone"))),
)

df_silver = df_silver.withColumnRenamed("wind.speed", "wind_ms")

df_silver = df_silver.withColumnRenamed("wind.deg", "wind_deg")

df_silver = df_silver.withColumnRenamed("main.temp", "temp_c")

df_silver = df_silver.withColumnRenamed("main.feels_like", "feels_like_c")

df_silver = df_silver.withColumnRenamed("main.temp_min", "temp_min_c")

df_silver = df_silver.withColumnRenamed("main.temp_max", "temp_max_c")

df_silver = df_silver.withColumnRenamed("main.pressure", "pressure_hpa")

df_silver = df_silver.withColumnRenamed("main.humidity", "humidity_pct")

df_silver = df_silver.withColumnRenamed("main.sea_level", "sea_level_hpa")

df_silver = df_silver.withColumnRenamed("main.grnd_level", "grnd_level_hpa")

df_silver = df_silver.withColumnRenamed("wind.gust", "wind_gust_ms")

df_silver = df_silver.withColumnRenamed("clouds.all", "clouds_pct")

df_silver = df_silver.withColumnRenamed("sys.country", "country")

df_silver = df_silver.withColumnRenamed("rain.1h", "rain_mm_h")

df_silver = df_silver.withColumnRenamed("coord.lat", "coord_lat")

df_silver = df_silver.withColumnRenamed("coord.lon", "coord_lon")

df_silver = df_silver.fillna({"rain_mm_h": 0})

df_silver = df_silver.withColumn("ingestion_timestamp", sf.current_timestamp())

df_silver = df_silver.select(
    "city",
    "coord_lon",
    "coord_lat",
    "temp_c",
    "feels_like_c",
    "temp_min_c",
    "temp_max_c",
    "pressure_hpa",
    "humidity_pct",
    "sea_level_hpa",
    "grnd_level_hpa",
    "wind_ms",
    "wind_deg",
    "wind_gust_ms",
    "clouds_pct",
    "country",
    "rain_mm_h",
    "weather_desc",
    "weather_main",
    "observation_timestamp_utc",
    "observation_timestamp_local",
    "sunrise_utc",
    "sunset_utc",
    "sunrise_local",
    "sunset_local",
    "ingestion_timestamp"
)

df_silver = df_silver.dropDuplicates(["observation_timestamp_utc", "city"])

if not DeltaTable.isDeltaTable(spark, silver_path):
    df_silver.write.mode("append").format("delta").save(silver_path)

else:
    delta_table = DeltaTable.forPath(spark, silver_path)
    (
        delta_table.alias("target")
        .merge(
            
            df_silver.alias("source"),
            """
            target.city = source.city
            AND target.observation_timestamp_utc = source.observation_timestamp_utc
            """,
        
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

display(df_silver)
df_silver.printSchema()